[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/04_rag_pipeline/04_rag_pipeline_solutions.ipynb)

# 04. RAG 파이프라인 실습 — 연습 문제 해설

[04_rag_pipeline.ipynb](04_rag_pipeline.ipynb) 끝의 연습 문제 2개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q numpy scikit-learn python-dotenv openai

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer

load_dotenv()
USE_OPENAI = bool(os.getenv("OPENAI_API_KEY"))
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

chunks = [
    {"source": "휴가규정.pdf", "page": 1,
     "text": "제2조(연차휴가) 1년간 80퍼센트 이상 출근한 임직원에게는 15일의 연차휴가를 부여한다. "
             "3년 이상 계속 근로한 임직원에 대하여는 매 2년마다 1일을 가산한 유급휴가를 부여한다."},
    {"source": "휴가규정.pdf", "page": 1,
     "text": "제3조(육아휴직) 만 8세 이하 자녀를 양육하는 임직원은 최대 1년의 육아휴직을 신청할 수 있다. "
             "육아휴직 기간은 근속연수에 포함하되, 승급을 위한 근속기간에는 산입하지 아니한다."},
    {"source": "휴가규정.pdf", "page": 2,
     "text": "제4조(경조휴가) 본인 결혼은 5일, 배우자 출산은 10일, 부모 사망은 5일의 유급휴가를 부여한다."},
    {"source": "휴가규정.pdf", "page": 2,
     "text": "제5조(병가) 업무 외 질병이나 부상으로 근로가 불가능한 경우 연간 60일 이내에서 병가를 "
             "사용할 수 있다. 병가 3일 이상 사용 시 진단서를 제출하여야 한다."},
    {"source": "복무규정.pdf", "page": 1,
     "text": "제10조(재택근무) 팀장의 승인을 받은 임직원은 주 2일까지 재택근무를 신청할 수 있다. "
             "재택근무 중 초과근무는 사전 승인 없이는 인정하지 않는다."},
]

if USE_OPENAI:
    from openai import OpenAI
    _client = OpenAI()

    def embed_texts(texts):
        response = _client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        return np.array([d.embedding for d in response.data])
else:
    _vectorizer = TfidfVectorizer()
    _vectorizer.fit([c["text"] for c in chunks])

    def embed_texts(texts):
        return _vectorizer.transform(texts).toarray()


chunk_vectors = embed_texts([c["text"] for c in chunks])


def cosine_similarity(a, b):
    a_norm = a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-8)
    b_norm = b / (np.linalg.norm(b, axis=-1, keepdims=True) + 1e-8)
    return a_norm @ b_norm.T


def search_similar_docs(question, k=4):
    query_vec = embed_texts([question])
    scores = cosine_similarity(query_vec, chunk_vectors)[0]
    top_k_idx = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in top_k_idx]

## 연습 1. `k`를 바꿔가며 결과 비교

In [ ]:
question = "육아휴직은 최대 몇 개월까지 쓸 수 있어?"

for k in [1, 3, 6]:
    print(f"--- k={k} ---")
    for doc, score in search_similar_docs(question, k=k):
        print(f"  [{score:.3f}] {doc['source']} p.{doc['page']} - {doc['text'][:35]}...")

**해설**
- `k=1`은 가장 관련도가 높은 조항(육아휴직 조항) 하나만 가져옵니다 — 답이 그 조항 안에 다 있다면 충분하지만, 여러 조항에 걸친 질문이면 정보가 부족할 수 있습니다.
- `k=6`처럼 corpus 전체 개수(5개)보다 크게 주면 있는 문서를 전부 가져오는 것과 같아집니다. 이 경우 경조휴가, 재택근무처럼 질문과 무관한 조항까지 프롬프트에 섞여 들어가서, 실제 LLM 호출 비용이 늘고 때로는 답변이 산만해질 수 있습니다.
- `query.py`가 `TOP_K = 4`로 적당히 작은 값을 쓰는 이유가 바로 이 트레이드오프(너무 적으면 정보 부족, 너무 많으면 비용/정확도 저하) 때문입니다.

## 연습 2. 문서에 없는 질문으로 grounding 확인

In [ ]:
out_of_scope_question = "해외 출장 비용 정산 기준이 뭐야?"

results = search_similar_docs(out_of_scope_question, k=3)
for doc, score in results:
    print(f"[{score:.3f}] {doc['source']} p.{doc['page']} - {doc['text'][:40]}...")

in_scope_question = "육아휴직은 최대 몇 개월까지 쓸 수 있어?"
in_scope_results = search_similar_docs(in_scope_question, k=3)
print("\n비교: 관련 있는 질문의 최고 점수 vs 관련 없는 질문의 최고 점수")
print("관련 있음:", round(in_scope_results[0][1], 3))
print("관련 없음:", round(results[0][1], 3))

**해설**
- corpus에 "해외 출장 비용"과 관련된 조항이 전혀 없으므로, 관련 없는 질문의 최고 유사도 점수가 관련 있는 질문보다 낮게 나옵니다 (그래도 corpus 안에서 "그나마 가장 가까운" 문서는 항상 반환된다는 점에 주의하세요 — 벡터 검색은 "이 안에 답이 있다/없다"를 스스로 판단하지 않고, 순위만 매깁니다).
- 이래서 `query.py`의 프롬프트에 "근거가 없으면 모른다고 답하라"는 지시가 중요합니다. 검색 자체는 항상 무언가를 반환하지만, 그 문서가 실제로 질문에 답할 근거가 되는지 판단하는 것은 최종적으로 LLM의 역할입니다.
- `OPENAI_API_KEY`가 설정되어 있다면 `answer(out_of_scope_question)`을 직접 호출해서, 관련 없는 조항이 검색되었을 때 모델이 실제로 "모른다"고 답하는지 확인해보세요.